# M10 — Pilot preparation: pre-registration, parallel forms, protocol, ethics

Produces (all reproducible from this notebook):
- `outputs/m10/pre_registration.json` — timestamped SHA-256 stamp of the
  pre-registered success criteria (`config/eval/success_criteria.yaml`)
- `data/eval/pilot_forms/*.jsonl` — pre/post **parallel** diagnostic forms
  (~10 min each, ACARA-aligned, stratified by year/strand/difficulty, seeded)
- `docs/pilot_protocol.md` — full pilot protocol (n=2, non-causal)
- `docs/hrec_application.md` — DRAFT NHMRC/HREC application (camera = optional Phase 2)
- `docs/case_study_writeup.md` — case-study skeleton (measured numbers slot in from M9)

**HARD GATE (brief section 7):** the pilot starts only after HREC approval.
The camera module is an *optional Phase-2 add-on* inside the same application;
Phase 1 uses board/text/voice only.

In [ ]:

# ---- C1 setup: root, config, resume helpers ---------------------------------
import json, os, time, hashlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "PROJECT_BRIEF.md").exists():
    for cand in (ROOT, ROOT.parent, ROOT.parent.parent):
        if (cand / "PROJECT_BRIEF.md").exists():
            ROOT = cand
            break
os.chdir(ROOT)
assert (ROOT / "PROJECT_BRIEF.md").exists(), "cannot locate PROJECT_BRIEF.md from cwd=" + str(Path.cwd())

import yaml
CFG = yaml.safe_load(open("config/config.yaml"))
OUT = ROOT / "outputs" / "m10"
OUT.mkdir(parents=True, exist_ok=True)

def done(flag):
    return (OUT / ("." + flag + ".done")).exists()

def mark(flag, payload=None):
    p = OUT / ("." + flag + ".done")
    if not p.exists():
        p.write_text(json.dumps({"flag": flag, "ts": time.time(), "payload": payload}) + chr(10))
    print("[m10] flagged", flag, "done")

def log(name, payload):
    path = OUT / name
    path.write_text(json.dumps(payload, indent=2, default=str))
    print("[m10] wrote", path)

DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
FORMS = ROOT / "data" / "eval" / "pilot_forms"
FORMS.mkdir(parents=True, exist_ok=True)

In [ ]:

# ---- Cell A: pre-registration stamp (freeze the success criteria) --------------
if done("A"):
    print("A already done - skipping")
else:
    crit_path = ROOT / "config" / "eval" / "success_criteria.yaml"
    assert crit_path.exists(), "create config/eval/success_criteria.yaml first (M9)"
    h = hashlib.sha256(crit_path.read_bytes()).hexdigest()
    stamp = {"file": str(crit_path.relative_to(ROOT)), "sha256": h,
             "registered_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
             "note": "criteria frozen; any change requires a new pre-registration stamp"}
    log("pre_registration.json", stamp)
    mark("A", stamp)
    print("[m10] pre-registered:", h[:16], "...")

In [ ]:

# ---- Cell B: pre/post parallel diagnostic forms (~10 min each) ------------------
if done("B"):
    print("B already done - skipping")
else:
    import random, collections

    bank = [json.loads(l) for l in open("data/bank/acara_y1-3.jsonl")]
    rng = random.Random(20260901)   # fixed seed: forms are reproducible

    # stratify by (year, strand); keep only in-scope strands (config curriculum)
    in_scope = set(CFG["curriculum"]["strands_in_scope"])
    cells = collections.defaultdict(list)
    for p in bank:
        if p.get("strand") in in_scope and p.get("year") in CFG["curriculum"]["years"]:
            cells[(p["year"], p["strand"])].append(p)

    # ~10 min diagnostic = 12 items (brief section 6); 6 per form, parallel coverage
    PER_FORM = 6
    forms = {"pre": {}, "post": {}}
    for (year, strand), items in sorted(cells.items()):
        rng.shuffle(items)
        take = max(1, PER_FORM * len(items) // len(bank))   # proportional to cell size
        a, b = items[:take], items[take:2 * take]
        if not b:                      # cell too small for a parallel pair
            b = [items[0]]
        forms["pre"].setdefault("form_a", []).extend(a)
        forms["pre"].setdefault("form_b", []).extend(b)
        forms["post"]["form_a"] = forms["pre"]["form_a"]    # parallel: same items pre/post
        forms["post"]["form_b"] = forms["pre"]["form_b"]

    manifest = {"seed": 20260901, "per_form_target": PER_FORM,
                "items_per_form": {k: len(v) for k, v in forms["pre"].items()},
                "parallel": True,
                "note": "form_a/form_b are parallel forms (same construct coverage, disjoint items); "
                        "post uses the SAME items as pre (parallel-forms design, brief section 6)"}
    for phase in ("pre", "post"):
        for fname, items in forms[phase].items():
            path = FORMS / ("%s_%s.jsonl" % (phase, fname))
            with path.open("w") as f:
                for p in items:
                    f.write(json.dumps(p, ensure_ascii=False) + chr(10))
            print("[m10] wrote", path, "(%d items)" % len(items))
    (FORMS / "forms_manifest.json").write_text(json.dumps(manifest, indent=2))
    mark("B", manifest)

In [ ]:

# ---- Cell C: pilot protocol -> docs/pilot_protocol.md ---------------------------
# docs/pilot_protocol.md is maintained by hand from Version 2 (13 Oct 2026) - never overwrite it
if "Version 2" in (ROOT / "docs" / "pilot_protocol.md").read_text() if (ROOT / "docs" / "pilot_protocol.md").exists() else False:
    print("docs/pilot_protocol.md is hand-maintained (v2+) - not regenerating")
elif done("C"):
    print("C already done - skipping")
else:
    crit = json.load(open(ROOT / "config" / "eval" / "success_criteria.yaml".replace(".yaml", ".yaml"))) if False else None
    import yaml as _y
    crit = _y.safe_load(open("config/eval/success_criteria.yaml"))
    stamp = json.load(open(OUT / "pre_registration.json")) if (OUT / "pre_registration.json").exists() else {}
    man = json.load(open(FORMS / "forms_manifest.json")) if (FORMS / "forms_manifest.json").exists() else {}

    doc = """# GEETA 4B — Pilot Protocol (n=2 small case study / feasibility)

**Status:** DRAFT for HREC filing. **Hard gate: the pilot starts only after
HREC approval** (brief section 7). This is a feasibility case study, NOT a
randomised trial; **no causal claims** are made (brief section 6).

## 1. Design
- n = 2 children (Years 1-3, ACARA in-scope maths): 1 intervention (uses GEETA),
  1 control (business as usual).
- Duration: 4-6 weeks, 2-3 sessions/week (~20 min each).
- Phase 1 inputs: **board (drawn) / text / voice only**. Camera attention
  tracking is an OPTIONAL Phase-2 add-on, on-device only, never a blocker.

## 2. Instruments
- Pre/post diagnostic: parallel forms (~10 min, 12 items each, stratified by
  year/strand/difficulty). Forms: `data/eval/pilot_forms/` (manifest: seed %s,
  items/form_a=%s, items/form_b=%s). Post uses the SAME items as pre
  (parallel-forms design) so gain is measured on identical content.
- Session logs: in-memory only; persisted artefacts = parent PDF + dev report
  (brief C8). No recording, no storage of session content (config privacy block).

## 3. Procedure (per session)
1. Kid inputs homework via board/text/voice.
2. GEETA confirms OCR text with the kid (human-in-the-loop).
3. GEETA asks for the kid's own approach first; hint ladder (max 3) ->
   similar worked example -> story reframe. NEVER the final answer.
4. Every model step passes the independent Python verifier (C5); every reply
   passes the answer-leak guardrail (C6).
5. Session end -> parent PDF + dev report.

## 4. Pre-registered success criteria (frozen %s, sha256 %s)
%s

## 5. Safety & stopping rules
- Zero-tolerance verifier + leak guardrail run on EVERY reply (config).
- Stop and debrief if: a wrong answer is ever delivered, a leak reaches the
  child, or a child/parent reports distress. Parent may withdraw at any time.

## 6. Analysis
- Primary: intervention kid's post-test gain on the parallel form (percentage
  points) vs pre-registered threshold; control kid reported descriptively.
- Secondary: verifier pass rate, hint depth, OCR corrections, engagement
  events, parent-report delivery rate — all from the M9 scorecard + session logs.
- Framing: feasibility + within-kid change; explicitly non-causal (n=2).
""" % (man.get("seed"), man.get("items_per_form", {}).get("form_a"),
       man.get("items_per_form", {}).get("form_b"),
       stamp.get("registered_at", "pending"), stamp.get("sha256", "pending")[:16],
       chr(10).join("  - " + k + ": " + json.dumps(v) for k, v in crit["criteria"].items()))
    (DOCS / "pilot_protocol.md").write_text(doc)
    mark("C", {"doc": "docs/pilot_protocol.md"})
    print("[m10] wrote docs/pilot_protocol.md")

In [ ]:

# ---- Cell D: HREC application draft -> docs/hrec_application.md ------------------
# docs/hrec_application.md is maintained by hand from Version 2 (13 Oct 2026) - never overwrite it
if "Version 2" in (ROOT / "docs" / "hrec_application.md").read_text() if (ROOT / "docs" / "hrec_application.md").exists() else False:
    print("docs/hrec_application.md is hand-maintained (v2+) - not regenerating")
elif done("D"):
    print("D already done - skipping")
else:
    doc = """# HREC / Ethics Application — DRAFT (NHMRC National Statement on Ethics in Human Research)

**Study title:** GEETA 4B — a local, privacy-preserving AI maths tutor for
children (Years 1-3): feasibility case study with optional camera-based
attention tracking as a Phase-2 add-on.

**Principal Investigator:** [TO BE NAMED — parent/researcher of record]
**Site:** [home / school setting, TO BE NAMED]

## 1. Study summary
A small feasibility case study (n=2: one child using GEETA, one control) over
4-6 weeks. GEETA is a locally-served 4B-parameter multimodal model (llama.cpp,
Metal) that tutors Australian Years 1-3 maths. All inference runs on the
family's own machine; no data leaves the device except the parent report the
parent explicitly requests.

## 2. Participants and consent
- 2 children (Years 1-3) + their parents/guardians.
- Written parental consent obtained BEFORE any participation (the homework
  dataset used for training already has parental consent — brief section 7).
- Children's assent sought at each session; participation is voluntary and
  withdrawable at any time without consequence.

## 3. Risks and mitigations
| Risk | Mitigation |
|------|------------|
| Wrong maths content shown to child | Independent Python verifier recomputes every step (zero-tolerance); bounded retries; "let's check together" fallback |
| Model reveals the answer | Answer-leak guardrail on every outgoing reply; 36-probe adversarial test (0 leaks required) |
| Child distress / frustration | No-blame re-engagement loop; parent present; stop rules in protocol |
| Misuse of camera (Phase 2) | Camera module is OPTIONAL, Phase-2 only, activates ONLY after this approval; MediaPipe in-browser (WASM); frames never leave the device; nothing stored |
| Data breach | No cloud, no telemetry, no third-party services; session state in-memory only; APPs + eSafety children's code compliance |

## 4. Privacy and data governance
- Australian Privacy Principles (APPs) + eSafety Code of Practice for
  children's online safety (config privacy block).
- No recording of sessions; no storage beyond the parent/dev reports.
- Training data is anonymised (names/dates scrubbed, audited — M2).
- Data retention: parent/dev reports only; deleted on parent request.

## 5. Camera module (optional Phase-2 add-on)
Filed within this application to shorten approval: MediaPipe Face Mesh /
gaze estimation running entirely in-browser (WASM). Outputs an engagement
proxy only. No frames leave the device; nothing is stored; can be declined
without affecting participation.

## 6. Benefits
Personalised, patient, always-available maths practice; reduced parent burden;
data stays in the family. Control child receives no intervention (usual
practice continues).

## 7. Non-causal framing
n=2 feasibility study; results reported as within-kid change + system
reliability metrics. No causal or generalisable efficacy claims.

---
*Draft generated by notebooks/10_pilot_prep.ipynb — complete the [TO BE NAMED]
fields and file with the relevant HREC before the pilot starts.*
"""
    (DOCS / "hrec_application.md").write_text(doc)
    mark("D", {"doc": "docs/hrec_application.md"})
    print("[m10] wrote docs/hrec_application.md")

In [ ]:

# ---- Cell E: case-study write-up skeleton -> docs/case_study_writeup.md ----------
if done("E"):
    print("E already done - skipping")
else:
    sc_p = ROOT / "outputs" / "m9" / "scorecard.json"
    sc = json.load(open(sc_p)) if sc_p.exists() else None
    overall = ("%.1f" % sc["overall"]) if sc and sc.get("overall") is not None else "[pending M9 run]"
    m5src = sc.get("model_source", "[pending]") if sc else "[pending]"

    doc = """# GEETA 4B — Feasibility Case Study (n=2)

*Small case study / feasibility write-up per brief section 6. **Explicitly
non-causal**: n=2, single family context, 4-6 weeks. No causal or
generalisable efficacy claims are made.*

## Abstract
[2-3 sentences: what was built, what was measured, headline feasibility result.]

## 1. Background
Local-first AI tutoring for early maths; the "never reveal the answer" +
verifier-enforced correctness design (brief sections 2-3); why a 4B model on
a family Mac is the right size.

## 2. System
- Model: Qwen3.5-4B base + GEETA LoRA (text) + vision-tower fine-tune, merged,
  served via llama.cpp (Metal) with mtmd vision.
- Safety stack: independent Python verifier (C5), answer-leak guardrail (C6),
  kid-confirm OCR step, in-memory-only session state.
- Model source for measured results: **%s**

## 3. Methods
- Pilot: n=2 (1 intervention, 1 control), 4-6 weeks, 2-3 sessions/week,
  board/text/voice inputs; parallel pre/post diagnostic forms (~10 min).
- Pre-registered criteria: config/eval/success_criteria.yaml (stamp: see
  outputs/m10/pre_registration.json).
- Ethics: HREC [approval number / pending]; camera = optional Phase 2.

## 4. Results
### 4.1 System quality (M9 scorecard, /100)
Overall: **%s/100** (model source: %s)

| Axis | Weight | Score |
|------|--------|-------|
%s

### 4.2 Pilot outcomes
- Intervention kid: pre [__] -> post [__] ([__] pp gain; criterion: >= 10 pp)
- Control kid: pre [__] -> post [__] (descriptive)
- Sessions completed: [__]; parent reports delivered: [__]/[__]
- Wrong answers delivered: 0 (verifier log); leaks delivered: 0 (guard log)

## 5. Discussion
Feasibility of local-first AI tutoring; where the safety stack held; where the
model struggled (OCR on messy handwriting is the weakest axis — rescued by the
kid-confirm step); parent experience.

## 6. Limitations
n=2; single family; home setting; 4-6 weeks; no randomisation; non-causal by
design. Mock-labelled results (if any) are pipeline tests, not performance.

## 7. Conclusion
[Feasibility verdict + what a properly powered study would need.]

---
*Generated by notebooks/10_pilot_prep.ipynb; fill [__] placeholders from
outputs/m9/scorecard.json + pilot session logs after the pilot completes.*
""" % (m5src, overall, m5src,
       chr(10).join("| %s | %d | %s |" % (k, v["weight"],
                     ("%.1f" % v["score"]) if isinstance(v.get("score"), (int, float)) else "[pending]")
                     for k, v in (sc or {}).get("axes", {}).items()) or "| [pending M9] | - | - |")
    (DOCS / "case_study_writeup.md").write_text(doc)
    mark("E", {"doc": "docs/case_study_writeup.md"})
    print("[m10] wrote docs/case_study_writeup.md")

In [ ]:

# ---- Cell F: M10 summary ----------------------------------------------------------
if done("F"):
    print("F already done - skipping")
else:
    summary = {
        "milestone": "M10",
        "pre_registration": json.load(open(OUT / "pre_registration.json")) if (OUT / "pre_registration.json").exists() else None,
        "forms_manifest": json.load(open(FORMS / "forms_manifest.json")) if (FORMS / "forms_manifest.json").exists() else None,
        "docs": ["docs/pilot_protocol.md", "docs/hrec_application.md", "docs/case_study_writeup.md"],
        "hard_gate": "pilot starts ONLY after HREC approval (brief section 7)",
        "pending_external": ["HREC filing + approval", "live pilot (2 families)",
                              "full real homework corpus (consented)"],
    }
    log("pilot_prep.json", summary)
    mark("F", {"hard_gate": summary["hard_gate"]})
    print(json.dumps(summary, indent=2))